<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_45_Copyright_Text_Similarity_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================
# EXPERIMENT 7
# COPYRIGHT TEXT SIMILARITY ANALYSIS
# Shingling, Jaccard Similarity, Cosine Similarity
# and Longest Common Word Run
# ==============================================================

import re
import math
from collections import Counter


# --------------------------------------------------------------
# 1. TEXT NORMALISATION
# --------------------------------------------------------------

def normalise(text):
    """
    Convert text to lowercase, remove punctuation,
    and collapse extra spaces.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# --------------------------------------------------------------
# 2. TOKENISATION
# --------------------------------------------------------------

def tokens(text):
    return normalise(text).split()


# --------------------------------------------------------------
# 3. K-WORD SHINGLING
# --------------------------------------------------------------

def shingles(text, k=5):
    """
    Create overlapping k-word sequences.
    """
    t = tokens(text)

    if not t:
        return set()

    if len(t) < k:
        return {" ".join(t)}

    return {
        " ".join(t[i:i+k])
        for i in range(len(t) - k + 1)
    }


# --------------------------------------------------------------
# 4. JACCARD SIMILARITY
# --------------------------------------------------------------

def jaccard(a, b, k=5):
    """
    Jaccard similarity between two sets of shingles.
    """
    A = shingles(a, k)
    B = shingles(b, k)

    if not A and not B:
        return 1.0

    if not A or not B:
        return 0.0

    return len(A & B) / len(A | B)


# --------------------------------------------------------------
# 5. COSINE SIMILARITY
# --------------------------------------------------------------

def cosine(a, b):
    """
    Cosine similarity using word-frequency vectors.
    """
    ca = Counter(tokens(a))
    cb = Counter(tokens(b))

    if not ca or not cb:
        return 0.0

    common = set(ca) & set(cb)

    numerator = sum(
        ca[word] * cb[word]
        for word in common
    )

    denominator_a = math.sqrt(
        sum(value ** 2 for value in ca.values())
    )

    denominator_b = math.sqrt(
        sum(value ** 2 for value in cb.values())
    )

    if denominator_a == 0 or denominator_b == 0:
        return 0.0

    return numerator / (denominator_a * denominator_b)


# --------------------------------------------------------------
# 6. LONGEST COMMON WORD RUN
# --------------------------------------------------------------

def longest_common_run(a, b):
    """
    Find the longest sequence of consecutive identical words.
    """
    A = tokens(a)
    B = tokens(b)

    if not A or not B:
        return 0

    best = 0
    previous = [0] * (len(B) + 1)

    for i in range(1, len(A) + 1):

        current = [0] * (len(B) + 1)

        for j in range(1, len(B) + 1):

            if A[i - 1] == B[j - 1]:
                current[j] = previous[j - 1] + 1
                best = max(best, current[j])

        previous = current

    return best


# --------------------------------------------------------------
# 7. VERDICT
# --------------------------------------------------------------

def verdict(a, b, k=5, jac_hi=0.30, run_hi=8):

    j = jaccard(a, b, k)
    c = cosine(a, b)
    r = longest_common_run(a, b)

    if j >= 0.80:
        label = "NEAR-IDENTICAL - likely verbatim reproduction"

    elif j >= jac_hi or r >= run_hi:
        label = "SUBSTANTIAL SIMILARITY - requires manual review"

    elif c >= 0.40:
        label = "TOPICAL OVERLAP ONLY - common subject vocabulary"

    else:
        label = "NO SIGNIFICANT SIMILARITY"

    return {
        "jaccard": round(j, 3),
        "cosine": round(c, 3),
        "longest_common_run": r,
        "verdict": label
    }


# ==============================================================
# TEST DOCUMENTS
# ==============================================================

ORIGINAL = (
    "The copyright in a computer programme subsists automatically "
    "upon its creation and registration is not a precondition for "
    "enforcement of the owner's rights."
)

VERBATIM = ORIGINAL

COSMETIC = (
    "the COPYRIGHT in a computer programme subsists automatically, "
    "upon its creation; and registration is NOT a precondition for "
    "enforcement of the owner's rights!!"
)

PARAPHRASE = (
    "Copyright over software arises the moment the work is created, "
    "and the author need not register it before enforcing the rights "
    "that the law confers."
)

UNRELATED = (
    "The switch forwards frames based on the destination MAC address "
    "stored in its content addressable memory table, flooding only "
    "when the entry is absent."
)


# ==============================================================
# DISPLAY COMPARISON RESULTS
# ==============================================================

print("=" * 75)
print("EXPERIMENT 7 - COPYRIGHT TEXT SIMILARITY ANALYSIS")
print("=" * 75)

print("\n1. VERBATIM / COSMETIC COPY")
result1 = verdict(ORIGINAL, COSMETIC)
print(result1)


print("\n2. PARAPHRASED TEXT")
result2 = verdict(ORIGINAL, PARAPHRASE)
print(result2)


print("\n3. UNRELATED TEXT")
result3 = verdict(ORIGINAL, UNRELATED)
print(result3)


# ==============================================================
# SHINGLE SIZE COMPARISON
# =============================================================

print("\n" + "=" * 75)
print("JACCARD SIMILARITY FOR DIFFERENT SHINGLE SIZES")
print("=" * 75)

for k in range(3, 9):

    score = jaccard(ORIGINAL, PARAPHRASE, k)

    print(
        f"k = {k}  ->  "
        f"Jaccard Similarity = {score:.3f}"
    )


# ==============================================================
# TEST CASES
# ==============================================================

def run_tests():

    results = []

    # TC1
    results.append((
        "TC1 identical text -> Jaccard 1.0",
        jaccard(ORIGINAL, VERBATIM) == 1.0
    ))

    # TC2
    results.append((
        "TC2 identical text -> cosine 1.0",
        abs(cosine(ORIGINAL, VERBATIM) - 1.0) < 1e-9
    ))

    # TC3
    results.append((
        "TC3 cosmetic edits detected",
        jaccard(ORIGINAL, COSMETIC) > 0.9
    ))

    # TC4
    results.append((
        "TC4 unrelated text has low Jaccard",
        jaccard(ORIGINAL, UNRELATED) < 0.05
    ))

    # TC5
    results.append((
        "TC5 paraphrase below cosmetic-copy score",
        jaccard(ORIGINAL, PARAPHRASE)
        < jaccard(ORIGINAL, COSMETIC)
    ))

    # TC6
    results.append((
        "TC6 cosmetic copy classified as near-identical",
        verdict(
            ORIGINAL,
            COSMETIC
        )["verdict"].startswith("NEAR-IDENTICAL")
    ))

    # TC7
    results.append((
        "TC7 unrelated text classified correctly",
        verdict(
            ORIGINAL,
            UNRELATED
        )["verdict"].startswith("NO SIGNIFICANT")
    ))

    # TC8
    results.append((
        "TC8 paraphrase classified as topical overlap",
        verdict(
            ORIGINAL,
            PARAPHRASE
        )["verdict"].startswith("TOPICAL OVERLAP")
    ))

    # TC9
    results.append((
        "TC9 longest common run detects copying",
        longest_common_run(
            ORIGINAL,
            COSMETIC
        ) >= 10
    ))

    # TC10
    results.append((
        "TC10 unrelated text has short common run",
        longest_common_run(
            ORIGINAL,
            UNRELATED
        ) <= 2
    ))

    # TC11
    results.append((
        "TC11 Jaccard is symmetric",
        abs(
            jaccard(ORIGINAL, PARAPHRASE)
            -
            jaccard(PARAPHRASE, ORIGINAL)
        ) < 1e-12
    ))

    # TC12
    results.append((
        "TC12 empty vs empty handled",
        jaccard("", "") == 1.0
    ))

    # TC13
    results.append((
        "TC13 empty vs text handled",
        jaccard("", ORIGINAL) == 0.0
    ))

    # Print results
    print("\n" + "=" * 75)
    print("TEST CASE RESULTS")
    print("=" * 75)

    for name, passed in results:
        print(
            f"{name:<50} -> "
            f"{'PASS' if passed else 'FAIL'}"
        )

    total_passed = sum(
        1 for _, passed in results if passed
    )

    print("\n" + "=" * 75)
    print(
        f"RESULT: {total_passed}/{len(results)} "
        "test cases passed"
    )
    print("=" * 75)

    return total_passed == len(results)


# ==============================================================
# RUN ALL TESTS
# ==============================================================

run_tests()

EXPERIMENT 7 - COPYRIGHT TEXT SIMILARITY ANALYSIS

1. VERBATIM / COSMETIC COPY
{'jaccard': 1.0, 'cosine': 1.0, 'longest_common_run': 24, 'verdict': 'NEAR-IDENTICAL - likely verbatim reproduction'}

2. PARAPHRASED TEXT
{'jaccard': 0.0, 'cosine': 0.423, 'longest_common_run': 1, 'verdict': 'TOPICAL OVERLAP ONLY - common subject vocabulary'}

3. UNRELATED TEXT
{'jaccard': 0.0, 'cosine': 0.311, 'longest_common_run': 1, 'verdict': 'NO SIGNIFICANT SIMILARITY'}

JACCARD SIMILARITY FOR DIFFERENT SHINGLE SIZES
k = 3  ->  Jaccard Similarity = 0.000
k = 4  ->  Jaccard Similarity = 0.000
k = 5  ->  Jaccard Similarity = 0.000
k = 6  ->  Jaccard Similarity = 0.000
k = 7  ->  Jaccard Similarity = 0.000
k = 8  ->  Jaccard Similarity = 0.000

TEST CASE RESULTS
TC1 identical text -> Jaccard 1.0                  -> PASS
TC2 identical text -> cosine 1.0                   -> PASS
TC3 cosmetic edits detected                        -> PASS
TC4 unrelated text has low Jaccard                 -> PASS
TC5 paraphr

True